# Statistical Testing for Structural Similarity Evaluation

This notebook computes uncertainty estimates and paired significance tests for the evaluation tasks used in the project: pairwise separation, retrieval, ranking, synthetic anchor comparison, SemEval 2026 triplets, and SemEval 2022 narrative-similarity correlation.

The key principle is that all uncertainty estimates are computed at the example level: pairs are resampled as pairs, retrieval/ranking examples are resampled as whole queries with all options kept together, and SemEval triplets/article pairs are resampled as intact examples. This avoids incorrectly treating multiple scores inside the same query as independent observations.

In [ ]:
from pathlib import Path
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    import torch
    import torch.nn.functional as F
    from transformers import AutoTokenizer, AutoModel
except Exception as e:
    torch = None
    F = None
    AutoTokenizer = None
    AutoModel = None
    print('[warn] Torch/transformers unavailable. You can still run stats if cached per-example outputs exist.')
    print('[warn]', e)

from scipy import stats
try:
    from statsmodels.stats.contingency_tables import mcnemar as statsmodels_mcnemar
except Exception:
    statsmodels_mcnemar = None

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'src' else cwd
print('PROJECT_ROOT:', PROJECT_ROOT)

RANDOM_SEED = 42
N_BOOT = 10_000
ALPHA = 0.05
FORCE_RECOMPUTE = False
INCLUDE_EXISTING_LLM_OUTPUTS = True

EVAL_DATA_DIR = PROJECT_ROOT / 'data' / 'eval_data'
EVAL_RESULTS_DIR = PROJECT_ROOT / 'data' / 'eval_results'
OUT_DIR = EVAL_RESULTS_DIR / 'statistical_testing'
OUT_DIR.mkdir(parents=True, exist_ok=True)

PER_EXAMPLE_PARQUET = OUT_DIR / 'embedding_per_example_outputs.parquet'
PER_EXAMPLE_CSV = OUT_DIR / 'embedding_per_example_outputs.csv'
POINT_TABLE_PATH = OUT_DIR / 'metric_bootstrap_ci_table.csv'
TEST_TABLE_PATH = OUT_DIR / 'paired_significance_tests.csv'
SUMMARY_TABLE_PATH = OUT_DIR / 'statistical_testing_summary.csv'

PATHS = {
    'eval_pair': EVAL_DATA_DIR / 'eval_story_pairs_200.csv',
    'retrieval': EVAL_DATA_DIR / 'retrieval_eval_df.csv',
    'ranking': EVAL_DATA_DIR / 'ranking_eval_df.csv',
    'synthetic_v2': EVAL_DATA_DIR / 'synthetic_v2_anchor_eval_data.csv',
    'semeval_2026': EVAL_DATA_DIR / 'SemEval2026-Task_4-dev-v1' / 'dev_track_a.jsonl',
    'semeval_2022': EVAL_DATA_DIR / 'SemEval2022-Task8' / 'semeval_2022_eval_data.csv',
}
for name, path in PATHS.items():
    print(f'{name:15s}: {path} | exists={path.exists()}')

## Model and Output Configuration

The statistical tests need per-example outputs from every model being compared. If cached per-example outputs already exist, the notebook loads them directly. Otherwise, it reconstructs them by loading the trained checkpoint and embedding baselines, running the same evaluation datasets, and saving a reusable cache.

The primary system is labeled `E5-Mistral+MSE`. Edit `OUR_CHECKPOINT_CANDIDATES` if the checkpoint lives somewhere else on a given machine.

In [ ]:
def first_existing(paths):
    for p in paths:
        p = Path(p).expanduser()
        if p.exists():
            return p
    return None

OUR_MODEL_NAME = 'E5-Mistral+MSE'
OUR_CHECKPOINT_CANDIDATES = [
    PROJECT_ROOT / 'artifacts' / 'e5_mistral_structural_finetune_v2' / 'checkpoints_mse_loss' / 'epoch_01_mse_loss',
    PROJECT_ROOT / 'artifacts' / 'e5_mistral_structural_finetune' / 'checkpoints_mse_loss' / 'epoch_01_mse_loss',
    Path('/tank/scratch/shayan/Projects/NarrativeSimilarity/artifacts/e5_mistral_structural_finetune_v2/checkpoints_mse_loss/epoch_01_mse_loss'),
    Path('/tank/scratch/shayan/Projects/NarrativeSimilarity/artifacts/e5_mistral_structural_finetune/checkpoints_mse_loss/epoch_01_mse_loss'),
    Path('/scratch/shayan/Projects/NarrativeSimilarity/artifacts/e5_mistral_structural_finetune_v2/checkpoints_mse_loss/epoch_01_mse_loss'),
    Path('/scratch/shayan/Projects/NarrativeSimilarity/artifacts/e5_mistral_structural_finetune/checkpoints_mse_loss/epoch_01_mse_loss'),
]
OUR_CHECKPOINT = first_existing(OUR_CHECKPOINT_CANDIDATES)
print('OUR_CHECKPOINT:', OUR_CHECKPOINT)

HF_CACHE_CANDIDATES = [
    PROJECT_ROOT / 'hf_cache',
    Path('/scratch/shayan/hf_cache'),
    Path('/tank/scratch/shayan/hf_cache'),
]
HF_CACHE_DIR = first_existing(HF_CACHE_CANDIDATES) or (PROJECT_ROOT / 'hf_cache')
print('HF_CACHE_DIR:', HF_CACHE_DIR)

MODEL_SPECS = [
    {
        'model': OUR_MODEL_NAME,
        'model_type': 'checkpoint',
        'path': OUR_CHECKPOINT,
        'use_e5_query_prefix': True,
        'max_length': 1024,
        'use_bfloat16': True,
    },
    {
        'model': 'bge_base',
        'model_type': 'embedding_baseline',
        'path': 'BAAI/bge-base-en-v1.5',
        'use_e5_query_prefix': False,
        'max_length': 1024,
        'use_bfloat16': True,
    },
    {
        'model': 'e5_untrained',
        'model_type': 'embedding_baseline',
        'path': 'intfloat/e5-mistral-7b-instruct',
        'use_e5_query_prefix': True,
        'max_length': 1024,
        'use_bfloat16': True,
    },
    {
        'model': 'roberta_base',
        'model_type': 'embedding_baseline',
        'path': 'roberta-base',
        'use_e5_query_prefix': False,
        'max_length': 1024,
        'use_bfloat16': True,
    },
    {
        'model': 'story_emb',
        'model_type': 'embedding_baseline',
        'path': 'uhhlt/story-emb',
        'use_e5_query_prefix': False,
        'max_length': 1024,
        'use_bfloat16': True,
    },
]

pd.DataFrame([{k: str(v) for k, v in spec.items()} for spec in MODEL_SPECS])

## Reconstruct Per-Example Outputs

This cell mirrors the evaluation logic from the CLI script, but stores one row per evaluation example instead of only aggregate metrics. These rows are what bootstrap confidence intervals and paired tests require.

The output cache is saved under `data/eval_results/statistical_testing/`. If the cache exists, set `FORCE_RECOMPUTE = False` to load it immediately.

In [ ]:
def read_jsonl(path: Path):
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


def parse_bool_label(v) -> bool:
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, np.integer)):
        return bool(v)
    if isinstance(v, str):
        vv = v.strip().lower()
        if vv in {'true', '1', 'yes', 'y', 'a', 'text_a', 'text_a_is_closer'}:
            return True
        if vv in {'false', '0', 'no', 'n', 'b', 'text_b'}:
            return False
    raise ValueError(f'Could not parse boolean label: {v!r}')


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    summed = (last_hidden_state * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-9)
    return summed / denom


class Embedder:
    def __init__(self, model_path, max_length=1024, use_bfloat16=True, use_e5_query_prefix=True):
        if torch is None or AutoTokenizer is None or AutoModel is None:
            raise RuntimeError('Torch/transformers are required to recompute embedding outputs.')
        if model_path is None:
            raise FileNotFoundError('Checkpoint path is None. Update OUR_CHECKPOINT_CANDIDATES or load an existing cache.')

        self.model_path = str(model_path)
        self.use_e5_query_prefix = bool(use_e5_query_prefix)
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        dtype = torch.bfloat16 if (self.device == 'cuda' and use_bfloat16) else torch.float32

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_path,
            trust_remote_code=True,
            cache_dir=str(HF_CACHE_DIR),
        )
        self.model = AutoModel.from_pretrained(
            self.model_path,
            trust_remote_code=True,
            cache_dir=str(HF_CACHE_DIR),
            dtype=dtype,
        )
        self.model.to(self.device)
        self.model.eval()

        tokenizer_cap = getattr(self.tokenizer, 'model_max_length', None)
        config_cap = getattr(getattr(self.model, 'config', None), 'max_position_embeddings', None)
        caps = [int(max_length)]
        if isinstance(tokenizer_cap, int) and 0 < tokenizer_cap < 10**6:
            caps.append(tokenizer_cap)
        if isinstance(config_cap, int) and 0 < config_cap < 10**6:
            caps.append(config_cap)
        self.max_length = int(min(caps))
        print(f'[Embedder] {self.model_path} | device={self.device} | max_length={self.max_length} | e5_prefix={self.use_e5_query_prefix}')

    def prepare_text(self, text: str) -> str:
        text = str(text)
        if self.use_e5_query_prefix:
            return f'query: retrieve stories with a similar narrative to the given story. {text}'
        return text

    @torch.no_grad()
    def embed(self, texts, batch_size=8):
        vectors = []
        for i in range(0, len(texts), batch_size):
            batch = list(texts[i:i + batch_size])
            enc = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors='pt',
            )
            enc = {k: v.to(self.device) for k, v in enc.items()}
            h = self.model(**enc).last_hidden_state
            pooled = mean_pool(h, enc['attention_mask'])
            pooled = F.normalize(pooled, p=2, dim=1)
            vectors.append(pooled.detach().float().cpu())
        return torch.cat(vectors, dim=0)

    def close(self):
        del self.model
        del self.tokenizer
        if torch is not None and torch.cuda.is_available():
            torch.cuda.empty_cache()


def cosine_pair(emb, a, b):
    ea, eb = emb.embed([emb.prepare_text(a), emb.prepare_text(b)], batch_size=2)
    return float(torch.dot(ea, eb).item())


def eval_pair_outputs(df, emb, model_name, model_type):
    rows = []
    for idx, r in enumerate(tqdm(df.itertuples(index=False), total=len(df), desc=f'{model_name}: pairwise', unit='pair')):
        sim = cosine_pair(emb, str(r.story_text_a), str(r.story_text_b))
        rows.append({
            'metric': 'pairwise_gap',
            'task': 'eval_pair_task',
            'model': model_name,
            'model_type': model_type,
            'example_id': str(getattr(r, 'pair_id', idx)),
            'dataset': getattr(r, 'dataset', None),
            'label': int(r.label),
            'score': sim,
        })
    return rows


def retrieval_outputs(df, emb, model_name, model_type):
    rows = []
    for idx, r in enumerate(tqdm(df.itertuples(index=False), total=len(df), desc=f'{model_name}: retrieval', unit='query')):
        q = emb.prepare_text(str(r.query_text))
        opts = [emb.prepare_text(str(getattr(r, f'option_{i}_text'))) for i in range(5)]
        eq = emb.embed([q], batch_size=1).squeeze(0)
        eo = emb.embed(opts, batch_size=5)
        sims = (eo @ eq).numpy()
        pred = int(np.argmax(sims))
        gt = int(r.correct_option_index)
        rows.append({
            'metric': 'retrieval_accuracy',
            'task': 'retrieval_task',
            'model': model_name,
            'model_type': model_type,
            'example_id': str(getattr(r, 'query_id', getattr(r, 'pair_id', idx))),
            'dataset': getattr(r, 'dataset', None),
            'correct': int(pred == gt),
            'pred': pred,
            'gold': gt,
        })
    return rows


def ndcg_at_5_from_ranks(sims, gold_ranks):
    rel = np.array([6 - int(x) for x in gold_ranks], dtype=np.float64)
    pred_order = np.argsort(-np.asarray(sims, dtype=np.float64))
    gains = rel[pred_order]
    discounts = 1.0 / np.log2(np.arange(2, 2 + len(gains), dtype=np.float64))
    dcg = float(np.sum(gains * discounts))
    ideal_order = np.argsort(-rel)
    idcg = float(np.sum(rel[ideal_order] * discounts))
    return float(dcg / idcg) if idcg > 0 else 0.0


def ranking_outputs(df, emb, model_name, model_type):
    rows = []
    for idx, r in enumerate(tqdm(df.itertuples(index=False), total=len(df), desc=f'{model_name}: ranking', unit='query')):
        q = emb.prepare_text(str(r.query_text))
        opts = [emb.prepare_text(str(getattr(r, f'option_{i}_text'))) for i in range(5)]
        eq = emb.embed([q], batch_size=1).squeeze(0)
        eo = emb.embed(opts, batch_size=5)
        sims = (eo @ eq).numpy()
        ranks = [int(getattr(r, f'option_{i}_rank')) for i in range(5)]
        ndcg = ndcg_at_5_from_ranks(sims, ranks)
        rows.append({
            'metric': 'ranking_ndcg',
            'task': 'ranking_task',
            'model': model_name,
            'model_type': model_type,
            'example_id': str(getattr(r, 'pair_id', idx)),
            'dataset': getattr(r, 'dataset', None),
            'ndcg': ndcg,
        })
    return rows


def synthetic_outputs(df, emb, model_name, model_type):
    rows = []
    for idx, r in enumerate(tqdm(df.itertuples(index=False), total=len(df), desc=f'{model_name}: synthetic', unit='theme')):
        s12 = float(r.struct_score_12)
        s13 = float(r.struct_score_13)
        if s12 == s13:
            continue
        gt = 'story_2' if s12 > s13 else 'story_3'
        e = emb.embed([emb.prepare_text(str(r.story_1)), emb.prepare_text(str(r.story_2)), emb.prepare_text(str(r.story_3))], batch_size=3)
        sim12 = float(torch.dot(e[0], e[1]).item())
        sim13 = float(torch.dot(e[0], e[2]).item())
        pred = 'story_2' if sim12 > sim13 else 'story_3'
        rows.append({
            'metric': 'synthetic_accuracy',
            'task': 'synthetic_task',
            'model': model_name,
            'model_type': model_type,
            'example_id': str(getattr(r, 'theme', idx)),
            'dataset': 'synthetic_v2',
            'correct': int(pred == gt),
            'pred': pred,
            'gold': gt,
        })
    return rows


def semeval_2026_outputs(df, emb, model_name, model_type):
    rows = []
    for idx, r in enumerate(tqdm(df.itertuples(index=False), total=len(df), desc=f'{model_name}: semeval2026', unit='triplet')):
        e = emb.embed([
            emb.prepare_text(str(r.anchor_text)),
            emb.prepare_text(str(r.text_a)),
            emb.prepare_text(str(r.text_b)),
        ], batch_size=3)
        sim_a = float(torch.dot(e[0], e[1]).item())
        sim_b = float(torch.dot(e[0], e[2]).item())
        pred = bool(sim_a > sim_b)
        gt = parse_bool_label(r.text_a_is_closer)
        rows.append({
            'metric': 'semeval_2026_accuracy',
            'task': 'semeval_2026_task',
            'model': model_name,
            'model_type': model_type,
            'example_id': str(idx),
            'dataset': 'semeval_2026_task_4',
            'correct': int(pred == gt),
            'pred': pred,
            'gold': gt,
        })
    return rows


def semeval_2022_outputs(df, emb, model_name, model_type):
    rows = []
    for idx, r in enumerate(tqdm(df.itertuples(index=False), total=len(df), desc=f'{model_name}: semeval2022', unit='pair')):
        sim = cosine_pair(emb, str(r.article_1), str(r.article_2))
        rows.append({
            'metric': 'semeval_2022_spearman',
            'task': 'semeval_2022_correlation_task',
            'model': model_name,
            'model_type': model_type,
            'example_id': str(getattr(r, 'pair_id', idx)),
            'dataset': 'semeval_2022_task_8',
            'score': sim,
            'human_NAR': float(r.NAR),
        })
    return rows


def load_eval_frames():
    frames = {}
    frames['eval_pair'] = pd.read_csv(PATHS['eval_pair'])
    frames['retrieval'] = pd.read_csv(PATHS['retrieval'])
    frames['ranking'] = pd.read_csv(PATHS['ranking'])
    frames['synthetic_v2'] = pd.read_csv(PATHS['synthetic_v2'])
    frames['semeval_2026'] = read_jsonl(PATHS['semeval_2026'])
    frames['semeval_2022'] = pd.read_csv(PATHS['semeval_2022'])
    return frames


def load_existing_llm_outputs():
    rows = []
    if not INCLUDE_EXISTING_LLM_OUTPUTS:
        return pd.DataFrame(rows)

    # Ranking GPT-5.4 outputs.
    p = EVAL_RESULTS_DIR / 'ranking_task_gpt54_predictions.csv'
    if p.exists():
        df = pd.read_csv(p)
        for r in df.itertuples(index=False):
            if pd.isna(getattr(r, 'ndcg_at_5')):
                continue
            rows.append({
                'metric': 'ranking_ndcg', 'task': 'ranking_task', 'model': str(r.model), 'model_type': 'llm_baseline',
                'example_id': str(r.pair_id), 'dataset': getattr(r, 'dataset', None), 'ndcg': float(r.ndcg_at_5),
            })

    # Synthetic GPT-5.4 outputs.
    p = EVAL_RESULTS_DIR / 'synthetic_anchor_eval_gpt54_predictions.csv'
    if p.exists():
        df = pd.read_csv(p)
        for idx, r in enumerate(df.itertuples(index=False)):
            rows.append({
                'metric': 'synthetic_accuracy', 'task': 'synthetic_task', 'model': 'gpt-5.4', 'model_type': 'llm_baseline',
                'example_id': str(getattr(r, 'theme', idx)), 'dataset': 'synthetic_v2', 'correct': int(bool(r.correct)),
                'pred': str(r.pred_higher_vs_story_1), 'gold': str(r.gt_higher_vs_story_1),
            })

    # SemEval 2026 GPT-5.4 outputs.
    p = EVAL_RESULTS_DIR / 'semeval_2026_gpt54_predictions.csv'
    if p.exists():
        df = pd.read_csv(p)
        for r in df.itertuples(index=False):
            rows.append({
                'metric': 'semeval_2026_accuracy', 'task': 'semeval_2026_task', 'model': str(r.model), 'model_type': 'llm_baseline',
                'example_id': str(r.row_index), 'dataset': 'semeval_2026_task_4', 'correct': int(bool(r.correct)),
                'pred': str(r.predicted_closer), 'gold': bool(r.gold_text_a_is_closer),
            })

    # SemEval 2022 GPT-5.4 NAR scores.
    p = EVAL_RESULTS_DIR / 'semeval_2022_task8_gpt54_nar_scores.csv'
    if p.exists():
        df = pd.read_csv(p)
        for r in df.itertuples(index=False):
            if pd.isna(getattr(r, 'llm_nar_score_1_to_4')):
                continue
            rows.append({
                'metric': 'semeval_2022_spearman', 'task': 'semeval_2022_correlation_task', 'model': str(r.model), 'model_type': 'llm_baseline',
                'example_id': str(r.pair_id), 'dataset': 'semeval_2022_task_8',
                'score': float(r.llm_nar_score_1_to_4), 'human_NAR': float(r.human_NAR),
            })

    return pd.DataFrame(rows)


def build_embedding_per_example_outputs(force_recompute=False):
    if not force_recompute:
        if PER_EXAMPLE_PARQUET.exists():
            print('Loading cached per-example outputs:', PER_EXAMPLE_PARQUET)
            return pd.read_parquet(PER_EXAMPLE_PARQUET)
        if PER_EXAMPLE_CSV.exists():
            print('Loading cached per-example outputs:', PER_EXAMPLE_CSV)
            return pd.read_csv(PER_EXAMPLE_CSV)

    frames = load_eval_frames()
    all_rows = []

    for spec in MODEL_SPECS:
        print('\n=== Evaluating', spec['model'], '===')
        emb = Embedder(
            spec['path'],
            max_length=spec['max_length'],
            use_bfloat16=spec['use_bfloat16'],
            use_e5_query_prefix=spec['use_e5_query_prefix'],
        )
        all_rows.extend(eval_pair_outputs(frames['eval_pair'], emb, spec['model'], spec['model_type']))
        all_rows.extend(retrieval_outputs(frames['retrieval'], emb, spec['model'], spec['model_type']))
        all_rows.extend(ranking_outputs(frames['ranking'], emb, spec['model'], spec['model_type']))
        all_rows.extend(synthetic_outputs(frames['synthetic_v2'], emb, spec['model'], spec['model_type']))
        all_rows.extend(semeval_2026_outputs(frames['semeval_2026'], emb, spec['model'], spec['model_type']))
        all_rows.extend(semeval_2022_outputs(frames['semeval_2022'], emb, spec['model'], spec['model_type']))
        emb.close()

    per_example = pd.DataFrame(all_rows)
    llm_rows = load_existing_llm_outputs()
    if len(llm_rows):
        per_example = pd.concat([per_example, llm_rows], ignore_index=True, sort=False)
        print('Included existing LLM output rows:', len(llm_rows))

    try:
        per_example.to_parquet(PER_EXAMPLE_PARQUET, index=False)
        print('Saved:', PER_EXAMPLE_PARQUET)
    except Exception as e:
        print('[warn] Could not save parquet:', e)
    per_example.to_csv(PER_EXAMPLE_CSV, index=False)
    print('Saved:', PER_EXAMPLE_CSV)
    return per_example


per_example_df = build_embedding_per_example_outputs(force_recompute=FORCE_RECOMPUTE)
print('Rows:', len(per_example_df))
display(per_example_df.groupby(['metric', 'model', 'model_type']).size().reset_index(name='n').sort_values(['metric', 'model_type', 'model']))

## Bootstrap Confidence Intervals

For each model and metric, this cell computes a 95\% percentile bootstrap confidence interval using 10,000 resamples with replacement. Resampling is done over whole examples: one pair for pairwise separation and SemEval 2022, one query for retrieval/ranking, one theme for synthetic, and one triplet for SemEval 2026.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

METRIC_DISPLAY_NAMES = {
    'pairwise_gap': 'Pairwise separation gap',
    'retrieval_accuracy': 'Retrieval accuracy',
    'ranking_ndcg': 'Ranking nDCG',
    'synthetic_accuracy': 'Synthetic accuracy',
    'semeval_2026_accuracy': 'SemEval 2026 accuracy',
    'semeval_2022_spearman': 'SemEval 2022 Spearman',
}


def metric_point_estimate(df):
    metric = df['metric'].iloc[0]
    if metric == 'pairwise_gap':
        pos = df.loc[df['label'].astype(int) == 1, 'score'].astype(float)
        neg = df.loc[df['label'].astype(int) == 0, 'score'].astype(float)
        return float(pos.mean() - neg.mean())
    if metric in {'retrieval_accuracy', 'synthetic_accuracy', 'semeval_2026_accuracy'}:
        return float(df['correct'].astype(float).mean())
    if metric == 'ranking_ndcg':
        return float(df['ndcg'].astype(float).mean())
    if metric == 'semeval_2022_spearman':
        if len(df) < 3:
            return np.nan
        return float(stats.spearmanr(df['score'].astype(float), df['human_NAR'].astype(float), nan_policy='omit').statistic)
    raise ValueError(f'Unknown metric: {metric}')


def bootstrap_ci(df, stat_fn, n_boot=N_BOOT, alpha=ALPHA, seed=RANDOM_SEED):
    local_rng = np.random.default_rng(seed)
    n = len(df)
    if n == 0:
        return np.nan, np.nan
    vals = np.empty(n_boot, dtype=np.float64)
    for b in range(n_boot):
        idx = local_rng.integers(0, n, size=n)
        vals[b] = stat_fn(df.iloc[idx])
    return tuple(np.nanpercentile(vals, [100 * alpha / 2, 100 * (1 - alpha / 2)]))


point_rows = []
for (metric, model), g in tqdm(per_example_df.groupby(['metric', 'model']), desc='bootstrap CIs', unit='model-metric'):
    g = g.dropna(how='all').reset_index(drop=True)
    point = metric_point_estimate(g)
    ci_low, ci_high = bootstrap_ci(g, metric_point_estimate, seed=abs(hash((metric, model, RANDOM_SEED))) % (2**32))
    point_rows.append({
        'metric': metric,
        'metric_name': METRIC_DISPLAY_NAMES.get(metric, metric),
        'model': model,
        'model_type': g['model_type'].iloc[0] if 'model_type' in g.columns else None,
        'n_examples': int(len(g)),
        'point_estimate': point,
        'ci_low': ci_low,
        'ci_high': ci_high,
        'ci_95': f'[{ci_low:.4f}, {ci_high:.4f}]',
    })

metric_ci_df = pd.DataFrame(point_rows).sort_values(['metric', 'model_type', 'model']).reset_index(drop=True)
metric_ci_df.to_csv(POINT_TABLE_PATH, index=False)
print('Saved:', POINT_TABLE_PATH)
display(metric_ci_df)

## Paired Significance Tests

This cell compares `E5-Mistral+MSE` against each available baseline on the same examples.

- Pairwise separation uses a paired bootstrap over the difference in positive-minus-negative gaps.
- Retrieval, synthetic, and SemEval 2026 accuracy use McNemar's test on paired correctness indicators.
- Ranking uses Wilcoxon signed-rank tests over paired per-query nDCG values.
- SemEval 2022 uses Steiger's dependent-correlation test, applied to rank-transformed variables so that the comparison matches Spearman correlation.

In [ ]:
def align_model_rows(df, metric, model_a, model_b):
    a = df[(df['metric'] == metric) & (df['model'] == model_a)].copy()
    b = df[(df['metric'] == metric) & (df['model'] == model_b)].copy()
    common = sorted(set(a['example_id'].astype(str)) & set(b['example_id'].astype(str)))
    a = a[a['example_id'].astype(str).isin(common)].sort_values('example_id').reset_index(drop=True)
    b = b[b['example_id'].astype(str).isin(common)].sort_values('example_id').reset_index(drop=True)
    if len(a) != len(b):
        raise RuntimeError(f'Alignment failed for {metric}: {model_a} vs {model_b}')
    return a, b


def paired_bootstrap_gap(our_df, base_df, n_boot=N_BOOT, alpha=ALPHA, seed=RANDOM_SEED):
    merged = our_df[['example_id', 'label', 'score']].merge(
        base_df[['example_id', 'label', 'score']],
        on=['example_id', 'label'],
        suffixes=('_our', '_baseline'),
    )
    if len(merged) == 0:
        return np.nan, np.nan, np.nan, np.nan

    def gap_diff(x):
        our_pos = x.loc[x['label'].astype(int) == 1, 'score_our'].astype(float)
        our_neg = x.loc[x['label'].astype(int) == 0, 'score_our'].astype(float)
        base_pos = x.loc[x['label'].astype(int) == 1, 'score_baseline'].astype(float)
        base_neg = x.loc[x['label'].astype(int) == 0, 'score_baseline'].astype(float)
        return float((our_pos.mean() - our_neg.mean()) - (base_pos.mean() - base_neg.mean()))

    observed = gap_diff(merged)
    local_rng = np.random.default_rng(seed)
    vals = np.empty(n_boot, dtype=np.float64)
    n = len(merged)
    for i in range(n_boot):
        vals[i] = gap_diff(merged.iloc[local_rng.integers(0, n, size=n)])
    ci_low, ci_high = np.nanpercentile(vals, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    p_boot = 2 * min(np.mean(vals <= 0), np.mean(vals >= 0))
    p_boot = float(min(1.0, p_boot))
    return observed, float(ci_low), float(ci_high), p_boot


def run_mcnemar(our_correct, base_correct):
    our = np.asarray(our_correct, dtype=bool)
    base = np.asarray(base_correct, dtype=bool)
    both_correct = int(np.sum(our & base))
    our_wrong_base_correct = int(np.sum((~our) & base))
    our_correct_base_wrong = int(np.sum(our & (~base)))
    both_wrong = int(np.sum((~our) & (~base)))
    table = [[both_correct, our_wrong_base_correct], [our_correct_base_wrong, both_wrong]]

    if statsmodels_mcnemar is not None:
        result = statsmodels_mcnemar(table, exact=True)
        return float(result.statistic), float(result.pvalue), table

    discordant = our_wrong_base_correct + our_correct_base_wrong
    if discordant == 0:
        return 0.0, 1.0, table
    p = stats.binomtest(min(our_wrong_base_correct, our_correct_base_wrong), n=discordant, p=0.5, alternative='two-sided').pvalue
    return float(min(our_wrong_base_correct, our_correct_base_wrong)), float(p), table


def run_wilcoxon(our_values, base_values):
    our = np.asarray(our_values, dtype=np.float64)
    base = np.asarray(base_values, dtype=np.float64)
    diffs = our - base
    if np.allclose(diffs, 0):
        return 0.0, 1.0
    res = stats.wilcoxon(our, base, zero_method='wilcox', alternative='two-sided')
    return float(res.statistic), float(res.pvalue)


def steiger_z_for_dependent_spearman(y, x1, x2):
    """Compare corr(y, x1) vs corr(y, x2) for overlapping dependent correlations.

    Spearman comparison is implemented by rank-transforming y, x1, and x2,
    then applying Steiger's dependent-correlation test to the Pearson
    correlations of the ranks.
    """
    y = np.asarray(y, dtype=np.float64)
    x1 = np.asarray(x1, dtype=np.float64)
    x2 = np.asarray(x2, dtype=np.float64)
    ok = np.isfinite(y) & np.isfinite(x1) & np.isfinite(x2)
    y, x1, x2 = y[ok], x1[ok], x2[ok]
    n = len(y)
    if n < 4:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    ry = stats.rankdata(y)
    rx1 = stats.rankdata(x1)
    rx2 = stats.rankdata(x2)
    r_y1 = float(np.corrcoef(ry, rx1)[0, 1])
    r_y2 = float(np.corrcoef(ry, rx2)[0, 1])
    r_12 = float(np.corrcoef(rx1, rx2)[0, 1])

    # Steiger/Williams test for two correlations sharing one variable.
    k = 1 - r_y1**2 - r_y2**2 - r_12**2 + 2 * r_y1 * r_y2 * r_12
    denom = math.sqrt(max(2 * k + ((r_y1 + r_y2) ** 2 / 4.0) * ((1 - r_12) ** 3), 1e-12))
    t_stat = (r_y1 - r_y2) * math.sqrt(max((n - 3) * (1 + r_12), 0.0)) / denom
    p_value = 2 * stats.t.sf(abs(t_stat), df=n - 3)
    return float(t_stat), float(p_value), r_y1, r_y2, r_12


test_rows = []
metrics = sorted(per_example_df['metric'].dropna().unique())
for metric in metrics:
    models = sorted(set(per_example_df.loc[per_example_df['metric'] == metric, 'model']) - {OUR_MODEL_NAME})
    for baseline in models:
        try:
            our, base = align_model_rows(per_example_df, metric, OUR_MODEL_NAME, baseline)
            if len(our) == 0:
                continue

            row = {
                'metric': metric,
                'metric_name': METRIC_DISPLAY_NAMES.get(metric, metric),
                'our_model': OUR_MODEL_NAME,
                'baseline_model': baseline,
                'n_paired_examples': int(len(our)),
            }

            if metric == 'pairwise_gap':
                diff, lo, hi, p = paired_bootstrap_gap(
                    our, base,
                    seed=abs(hash((metric, baseline, 'gap', RANDOM_SEED))) % (2**32),
                )
                row.update({
                    'test_name': 'paired bootstrap gap difference',
                    'test_statistic': diff,
                    'p_value': p,
                    'difference': diff,
                    'difference_ci_low': lo,
                    'difference_ci_high': hi,
                    'difference_ci_95': f'[{lo:.4f}, {hi:.4f}]',
                    'significant_alpha_0_05': bool(lo > 0 or hi < 0),
                })
            elif metric in {'retrieval_accuracy', 'synthetic_accuracy', 'semeval_2026_accuracy'}:
                stat, p, table = run_mcnemar(our['correct'].astype(int), base['correct'].astype(int))
                row.update({
                    'test_name': 'McNemar exact test',
                    'test_statistic': stat,
                    'p_value': p,
                    'mcnemar_table': str(table),
                    'significant_alpha_0_05': bool(p < ALPHA),
                })
            elif metric == 'ranking_ndcg':
                stat, p = run_wilcoxon(our['ndcg'].astype(float), base['ndcg'].astype(float))
                diff = float(our['ndcg'].astype(float).mean() - base['ndcg'].astype(float).mean())
                row.update({
                    'test_name': 'Wilcoxon signed-rank test',
                    'test_statistic': stat,
                    'p_value': p,
                    'difference': diff,
                    'significant_alpha_0_05': bool(p < ALPHA),
                })
            elif metric == 'semeval_2022_spearman':
                merged = our[['example_id', 'human_NAR', 'score']].merge(
                    base[['example_id', 'score']],
                    on='example_id',
                    suffixes=('_our', '_baseline'),
                )
                t_stat, p, r_our, r_base, r_between = steiger_z_for_dependent_spearman(
                    merged['human_NAR'], merged['score_our'], merged['score_baseline']
                )
                row.update({
                    'test_name': 'Steiger dependent Spearman test',
                    'test_statistic': t_stat,
                    'p_value': p,
                    'our_spearman': r_our,
                    'baseline_spearman': r_base,
                    'model_score_correlation': r_between,
                    'difference': float(r_our - r_base),
                    'significant_alpha_0_05': bool(p < ALPHA) if np.isfinite(p) else False,
                })
            else:
                continue
            test_rows.append(row)
        except Exception as e:
            warnings.warn(f'Failed {metric}: {OUR_MODEL_NAME} vs {baseline}: {e}')

test_df = pd.DataFrame(test_rows).sort_values(['metric', 'baseline_model']).reset_index(drop=True)
test_df.to_csv(TEST_TABLE_PATH, index=False)
print('Saved:', TEST_TABLE_PATH)
display(test_df)

## Final Reporting Table

This table combines each model's point estimate and bootstrap confidence interval with the paired significance result against `E5-Mistral+MSE` whenever that comparison is defined. For the primary model itself, test columns are left blank because it is the reference system.

In [ ]:
summary = metric_ci_df.copy()
summary['point_with_ci'] = summary.apply(
    lambda r: f"{r['point_estimate']:.4f} {r['ci_95']}", axis=1
)

comparison_cols = [
    'metric', 'baseline_model', 'test_name', 'test_statistic', 'p_value',
    'difference', 'difference_ci_95', 'significant_alpha_0_05',
    'n_paired_examples', 'mcnemar_table',
]
comparison_cols = [c for c in comparison_cols if c in test_df.columns]
comp = test_df[comparison_cols].rename(columns={'baseline_model': 'model'}) if len(test_df) else pd.DataFrame(columns=['metric', 'model'])

summary = summary.merge(comp, on=['metric', 'model'], how='left')
summary = summary[[
    'metric_name', 'metric', 'model_type', 'model', 'n_examples',
    'point_estimate', 'ci_95', 'test_name', 'test_statistic', 'p_value',
    'difference', 'difference_ci_95', 'significant_alpha_0_05', 'n_paired_examples'
] if 'n_paired_examples' in summary.columns else [
    'metric_name', 'metric', 'model_type', 'model', 'n_examples',
    'point_estimate', 'ci_95', 'test_name', 'test_statistic', 'p_value',
    'difference', 'difference_ci_95', 'significant_alpha_0_05'
]]
summary = summary.sort_values(['metric', 'model_type', 'model']).reset_index(drop=True)
summary.to_csv(SUMMARY_TABLE_PATH, index=False)
print('Saved:', SUMMARY_TABLE_PATH)
display(summary)

## Notes for Paper Reporting

Use `metric_bootstrap_ci_table.csv` for confidence intervals around point estimates and `paired_significance_tests.csv` for direct model-vs-baseline tests. The combined `statistical_testing_summary.csv` is convenient for paper tables.

Interpretation conventions:

- For pairwise separation, positive differences mean `E5-Mistral+MSE` has a larger positive-minus-negative cosine gap than the baseline.
- For accuracy and nDCG, positive differences mean `E5-Mistral+MSE` performs better on average.
- For SemEval 2022, Spearman correlation is the primary metric because NAR is ordinal. The Steiger test is applied after rank-transforming model scores and human NAR labels.